A｜数据审计

In [1]:
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, log_loss, confusion_matrix
from sklearn.base import clone
DATA=Path('../data/heart.csv')
if not DATA.exists(): DATA=Path('08-linear-models/data/heart.csv')
df=pd.read_csv(DATA)
target='heart_disease_present'
features=['age','sex','cp','trestbps','chol','fbs','restecg','thalach','exang','oldpeak','slope','ca','thal']
X=df[features].copy(); y=df[target].astype(int)
X_dev,X_test,y_dev,y_test=train_test_split(X,y,test_size=.2,stratify=y,random_state=2026)
print(df.shape, y.value_counts().to_dict(), X_dev.shape, X_test.shape)
print(df.isna().sum().to_string())
print('Development class counts:',y_dev.value_counts().to_dict())
print('num matches outcome:',bool(((df.num>0).astype(int)==y).all()))


(270, 18) {0: 150, 1: 120} (216, 13) (54, 13)
sample_id                  0
record_id                  0
site                       0
age                        0
sex                        0
cp                         0
trestbps                  22
chol                       9
fbs                       25
restecg                    0
thalach                   21
exang                     21
oldpeak                   24
slope                    103
ca                       181
thal                     151
num                        0
heart_disease_present      0
Development class counts: {0: 120, 1: 96}
num matches outcome: True


B｜开发集折外比较

In [2]:
num_cols=['age','trestbps','chol','thalach','oldpeak']
cat_cols=[c for c in features if c not in num_cols]
pre=ColumnTransformer([('num',Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler())]),num_cols),('cat',Pipeline([('impute',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore', sparse_output=False))]),cat_cols)])
models={'logistic':LogisticRegression(max_iter=3000),'lda':LinearDiscriminantAnalysis(),'gaussian_nb':GaussianNB()}
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=2026)
def metric_row(name, p, yy, t=.5):
    pred=(p>=t).astype(int); tn,fp,fn,tp=confusion_matrix(yy,pred,labels=[0,1]).ravel()
    return {'model':name,'roc_auc':roc_auc_score(yy,p),'pr_auc':average_precision_score(yy,p),'brier':brier_score_loss(yy,p),'log_loss':log_loss(yy,p),'sensitivity':tp/(tp+fn) if tp+fn else np.nan,'specificity':tn/(tn+fp) if tn+fp else np.nan,'FN':fn,'FP':fp,'cost_5FN_FP':5*fn+fp}
rows=[]
for name,est in models.items():
    pipe=Pipeline([('pre',pre),('model',est)])
    p=cross_val_predict(pipe,X_dev,y_dev,cv=cv,method='predict_proba')[:,1]
    rows.append(metric_row(name,p,y_dev))
prior=y_dev.mean(); p_base=np.empty(len(y_dev));
for tr,va in cv.split(X_dev,y_dev): p_base[va]=y_dev.iloc[tr].mean()
rows.append(metric_row('prevalence_baseline',p_base,y_dev) | {'roc_auc':0.5, 'pr_auc':prior})
results=pd.DataFrame(rows).sort_values('cost_5FN_FP'); results
print(results.round(4).to_string(index=False))

              model  roc_auc  pr_auc  brier  log_loss  sensitivity  specificity  FN  FP  cost_5FN_FP
           logistic   0.8314  0.7670 0.1723    0.5303       0.6458       0.8083  34  23          193
                lda   0.8101  0.7581 0.1923    0.6193       0.6146       0.7833  37  26          211
        gaussian_nb   0.7797  0.7456 0.2680    3.5858       0.4792       0.8833  50  14          264
prevalence_baseline   0.5000  0.4444 0.2469    0.6870       0.0000       1.0000  96   0          480


C1｜代价与阈值

In [3]:
chosen_name='logistic'  # 先依据假设和目标写出理由，再修改此处。
if chosen_name=='prevalence_baseline': p_dev=np.repeat(prior,len(y_dev))
else:
    pipe=Pipeline([('pre',pre),('model',models[chosen_name])]); p_dev=cross_val_predict(pipe,X_dev,y_dev,cv=cv,method='predict_proba')[:,1]
threshold_rows=[metric_row(chosen_name,p_dev,y_dev,t) | {'threshold':t} for t in [.1,.2,.3,.4,.5,.6,.7,.8,.9]]
thresholds=pd.DataFrame(threshold_rows).sort_values(['cost_5FN_FP','threshold']); thresholds
thresholds['TP']=int(y_dev.sum())-thresholds.FN
thresholds['TN']=int((1-y_dev).sum())-thresholds.FP
thresholds['precision']=thresholds.TP/(thresholds.TP+thresholds.FP)
print(thresholds.round(4).to_string(index=False))


   model  roc_auc  pr_auc  brier  log_loss  sensitivity  specificity  FN  FP  cost_5FN_FP  threshold  TP  TN  precision
logistic   0.8314   0.767 0.1723    0.5303       0.9792       0.3167   2  82           92        0.1  94  38     0.5341
logistic   0.8314   0.767 0.1723    0.5303       0.8958       0.5583  10  53          103        0.2  86  67     0.6187
logistic   0.8314   0.767 0.1723    0.5303       0.8438       0.7000  15  36          111        0.3  81  84     0.6923
logistic   0.8314   0.767 0.1723    0.5303       0.7188       0.7667  27  28          163        0.4  69  92     0.7113
logistic   0.8314   0.767 0.1723    0.5303       0.6458       0.8083  34  23          193        0.5  62  97     0.7294
logistic   0.8314   0.767 0.1723    0.5303       0.5938       0.8333  39  20          215        0.6  57 100     0.7403
logistic   0.8314   0.767 0.1723    0.5303       0.4896       0.9000  49  12          257        0.7  47 108     0.7966
logistic   0.8314   0.767 0.1723    0.53

C2｜系数稳定性与假设压力

In [4]:
log_pipe=Pipeline([('pre',pre),('model',LogisticRegression(max_iter=3000))]).fit(X_dev,y_dev)
names=log_pipe.named_steps['pre'].get_feature_names_out(); coef=log_pipe.named_steps['model'].coef_[0]
coef_table=pd.DataFrame({'feature':names,'beta':coef,'OR':np.exp(coef)}).sort_values('OR',ascending=False); coef_table.head(10)
print('Encoded dimensions:', len(names))
print(coef_table.round(4).to_string(index=False))
rng=np.random.default_rng(2026)
X_stress=X_dev.assign(age_repeat=X_dev.age+rng.normal(0,.25,len(X_dev)))
pre_stress=clone(pre)
spec=pre_stress.transformers
spec[0]=(spec[0][0],spec[0][1],num_cols+['age_repeat'])
stress=Pipeline([('pre',pre_stress),('model',clone(models['logistic']))])
for xx, model in [(X_dev,log_pipe),(X_stress,stress)]:
    pp=cross_val_predict(clone(model),xx,y_dev,cv=cv,method='predict_proba')[:,1]
    boot=[]
    for _ in range(300):
        idx=rng.integers(0,len(xx),len(xx))
        fit=clone(model).fit(xx.iloc[idx],y_dev.iloc[idx])
        co=pd.Series(fit.named_steps['model'].coef_[0],index=fit.named_steps['pre'].get_feature_names_out())
        boot.append(co.reindex(['num__age','num__age_repeat','cat__cp_4.0']))
    print('Age stress:',xx.shape[1],metric_row('logistic',pp,y_dev))
    print(pd.DataFrame(boot).quantile([.025,.5,.975]).round(4).to_string())
zz=pd.DataFrame(pre.fit_transform(X_dev),columns=pre.get_feature_names_out(),index=X_dev.index)[['num__'+c for c in num_cols]]
a,b=zz[y_dev==0],zz[y_dev==1]
print('Skew:',pd.DataFrame({'negative':a.skew(),'positive':b.skew()}).round(3))
print('Relative covariance difference:',np.linalg.norm(a.cov()-b.cov())/np.linalg.norm(zz.cov()))
# Deterministic monotone skew stress on cholesterol, with the same fold assignments.
xx=X_dev.assign(chol=np.exp(X_dev.chol/100))
for model in ['lda','logistic']:
    for tag,x in [('original',X_dev),('skew',xx)]:
        p=cross_val_predict(Pipeline([('pre',clone(pre)),('model',clone(models[model]))]),x,y_dev,cv=cv,method='predict_proba')[:,1]
        print(tag,metric_row(model,p,y_dev))


Encoded dimensions: 28
         feature    beta     OR
     cat__cp_4.0  0.9486 2.5820
     cat__ca_3.0  0.7537 2.1248
    num__oldpeak  0.6005 1.8230
  cat__slope_2.0  0.5659 1.7610
   cat__thal_7.0  0.5435 1.7220
    cat__sex_1.0  0.4389 1.5511
     cat__ca_1.0  0.3718 1.4504
  cat__exang_1.0  0.3527 1.4228
    cat__fbs_1.0  0.3516 1.4213
        num__age  0.3054 1.3572
cat__restecg_1.0  0.2997 1.3495
   cat__thal_6.0  0.0726 1.0753
     cat__ca_2.0  0.0101 1.0101
   num__trestbps -0.0182 0.9820
cat__restecg_0.0 -0.0691 0.9332
     cat__cp_1.0 -0.0847 0.9188
    num__thalach -0.1338 0.8748
  cat__slope_3.0 -0.1379 0.8711
cat__restecg_2.0 -0.2271 0.7969
    cat__fbs_0.0 -0.3480 0.7061
  cat__exang_0.0 -0.3491 0.7053
     cat__cp_3.0 -0.3865 0.6794
  cat__slope_1.0 -0.4243 0.6542
    cat__sex_0.0 -0.4354 0.6470
     cat__cp_2.0 -0.4737 0.6227
       num__chol -0.4871 0.6144
   cat__thal_3.0 -0.6124 0.5420
     cat__ca_0.0 -1.1320 0.3224
Age stress: 13 {'model': 'logistic', 'roc_auc': 0

## D｜冻结协议
主模型采用课程 Notebook 的 L2 logistic。按训练折中位数填补并标准化五个连续变量，其余八列众数填补并完整 one-hot。使用 5 折分层开发集折外预测，在 0.1 至 0.9 的预定网格上最小化 5FN+FP，并在并列时选较低阈值。冻结指标为 ROC-AUC、AP、Brier、log loss、混淆矩阵、敏感度、特异度、精确率及代价。采用按结局分层的测试记录 bootstrap（2000 次）估计固定模型指标的百分位区间。

In [5]:
OPEN_LOCKED_TEST = True  # 已按上文冻结协议完成留出评价

chosen=float(thresholds.iloc[0].threshold)
print('Frozen threshold:',chosen)
if OPEN_LOCKED_TEST:
    final_model=clone(log_pipe).fit(X_dev,y_dev)
    p_test=final_model.predict_proba(X_test)[:,1]
    yy=y_test.to_numpy()
    def evaluate(p,y):
        row=metric_row('logistic',p,y,chosen)
        tp=int(y.sum())-row['FN']; tn=int((1-y).sum())-row['FP']
        return row | {'TN':tn,'TP':tp,'precision':tp/(tp+row['FP'])}
    print(evaluate(p_test,yy))
    rng=np.random.default_rng(2026)
    groups=[np.flatnonzero(yy==k) for k in (0,1)]
    boot=[]
    for _ in range(2000):
        i=np.concatenate([rng.choice(g,len(g),replace=True) for g in groups])
        boot.append(evaluate(p_test[i],yy[i]))
    print(pd.DataFrame(boot).drop(columns='model').quantile([.025,.975]).round(4).T.to_string())


Frozen threshold: 0.1
{'model': 'logistic', 'roc_auc': 0.8472222222222222, 'pr_auc': 0.7972915296338456, 'brier': 0.16325840217119542, 'log_loss': 0.48305500625782716, 'sensitivity': np.float64(1.0), 'specificity': np.float64(0.4), 'FN': np.int64(0), 'FP': np.int64(18), 'cost_5FN_FP': np.int64(18), 'TN': np.int64(12), 'TP': np.int64(24), 'precision': np.float64(0.5714285714285714)}
               0.025    0.975
roc_auc       0.7347   0.9333
pr_auc        0.6591   0.9223
brier         0.1092   0.2276
log_loss      0.3452   0.6499
sensitivity   1.0000   1.0000
specificity   0.2333   0.5667
FN            0.0000   0.0000
FP           13.0000  23.0000
cost_5FN_FP  13.0000  23.0000
TN            7.0000  17.0000
TP           24.0000  24.0000
precision     0.5106   0.6486
